## Terminology Update

Some historical output logs in this notebook were captured before the current naming migration.
Current runtime terminology in the codebase is:
- `single_csv`
- `multi_csv`
- `ExecutionContext`


In [1]:
# Setup: add project root
import sys
import os
sys.path.insert(0, '..')

from src.orchestrator import Orchestrator
from src.standards import METADATA_STANDARDS
from src.context.context_factory import create_context
BASE = os.path.abspath(os.path.join('..'))

import logging

# Setup logging for Jupyter notebook
# Force reconfigure by removing existing handlers
logger = logging.getLogger()
logger.setLevel(logging.INFO)

# Remove existing handlers to avoid duplicates
for handler in logger.handlers[:]:
    logger.removeHandler(handler)

# Add a fresh StreamHandler that outputs to stdout (visible in notebook)
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(name)s - %(message)s'))
logger.addHandler(handler)

print("Imports OK")

Python-dotenv could not parse statement starting at line 5


Imports OK


In [2]:
from src.orchestrator import run_metadata_extraction
multi_source = {
    'biota': os.path.join(BASE, 'data/biota_dataset/biota.csv'),
    'samples': os.path.join(BASE, 'data/biota_dataset/samples.csv'),
    'species': os.path.join(BASE, 'data/biota_dataset/species.csv'),
}

# multi_source = {
#     'event': os.path.join(BASE, 'data/protoDT/annual_budburst_per_tree.csv'),
#     'occurrence': os.path.join(BASE, 'data/protoDT/budburst_climwin_input.csv'),
#     'temp': os.path.join(BASE, 'data/protoDT/temp_climwin_input.csv'),
# }

# multi_source = {
#     'event': os.path.join(BASE, 'data/cropharvest/cropharvest_cleaned_global.csv')
# }

# multi_source = {
#     'event': os.path.join(BASE, 'data/S2BMS/bms_presence_y-2018-2019_th-200.csv'),
# }

In [3]:
data_context = create_context(
    source=multi_source,
    name='biota_multi'
)
metadata_standard=METADATA_STANDARDS['spatial_ecological']
orchestrator = Orchestrator(topology_name='fast')
plan = orchestrator.generate_plan(
    context=data_context,
    metadata_standard=metadata_standard
)

/tmp/ipykernel_238599/4072587781.py:6: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  orchestrator = Orchestrator(topology_name='fast')


2026-05-28 14:30:35,397 - INFO - root - PlanExecutor initialized with topology: fast
2026-05-28 14:30:35,398 - INFO - root -   Players per step: 2
2026-05-28 14:30:35,398 - INFO - root -   Debate rounds: 1
2026-05-28 14:30:35,399 - INFO - root -   Player pool: ['data_analyst', 'schema_expert']
2026-05-28 14:30:35,399 - INFO - root - Orchestrator initialized with topology: fast
2026-05-28 14:30:35,399 - INFO - root - ============================================================
2026-05-28 14:30:35,399 - INFO - root - GENERATING PLAN
2026-05-28 14:30:35,399 - INFO - root - Context: biota_multi
2026-05-28 14:30:35,400 - INFO - root - Context type: multi_csv
2026-05-28 14:30:35,400 - INFO - root - Classified planning type: multi_csv
2026-05-28 14:30:35,400 - INFO - root - Resources: ['biota', 'samples', 'species']
2026-05-28 14:30:35,400 - INFO - root - Is multi-context: True
2026-05-28 14:30:35,400 - INFO - root - ============================================================
2026-05-28 14:3

/home/com3dian/Github/metadata_agent/notebooks/../src/players/player.py:1150: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return Player(


2026-05-28 14:31:02,978 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-05-28 14:31:03,011 - WARNING - root - Plan validation warning: Step 3 ('Generate complete metadata record with concrete values for all standard fields using gathered statistics, relationships, and spatial-temporal extents') has unmet dependencies. Missing artifacts: {'metadata_standard'}
2026-05-28 14:31:03,012 - INFO - root - Plan generated successfully!
2026-05-28 14:31:03,012 - INFO - root - Number of steps: 4
2026-05-28 14:31:03,013 - INFO - root -   Step 1: Perform comprehensive field statistics and missing value analysis for all resources to establish data distributions and quality metrics (player: data_analyst) (resources: ['biota', 'samples', 'species'])
2026-05-28 14:31:03,013 - INFO - root -   Step 2: Discover and validate relationships between resources using column name patterns and data type compatibility (player: relationship_analyst) (resource

In [5]:
# Validate plan dataflow
from src.orchestrator.utils import validate_plan_dataflow

if plan:
    # Convert to dict list for validation
    plan_dicts = plan.to_dict_list()
    
    is_valid, message = validate_plan_dataflow(plan_dicts)
    
    if is_valid:
        print(f"✅ {message}")
    else:
        print(f"❌ {message}")
else:
    print("No plan to validate.")

✅ Plan dataflow is valid.


In [6]:
plan.pretty_print()

Plan Steps:
Step 0: Perform comprehensive field statistics and missing value analysis for all resources to establish data distributions and quality metrics
  Rationale: Need field statistics and missing value counts for all three resources (biota, samples, species) to provide concrete data distributions for metadata generation. Combining all resources in one step for efficiency.
  Required Artifacts: {}
  Produced Artifacts: ['biota:field_stats', 'biota:missing_values', 'samples:field_stats', 'samples:missing_values', 'species:field_stats', 'species:missing_values']
Step 1: Discover and validate relationships between resources using column name patterns and data type compatibility
  Rationale: Need to confirm the discovered relationships (biota.sample_id -> samples.sample_id and biota.sibes_id -> species.sibes_id) to understand the data model structure for metadata generation.
  Required Artifacts: {}
  Produced Artifacts: ['relationships']
Step 2: Analyze spatial and temporal coverage

In [7]:
from src.orchestrator.plan_executor import PlanExecutor
from src.tools.context_tools import register_context

context_key = "ctx_biota_multi"
register_context(context_key, data_context)
metadata_standard = METADATA_STANDARDS["spatial_ecological"]
executor = PlanExecutor(topology_name="single")
result = executor.execute(
    plan=plan,
    context=data_context,
    context_key=context_key,
    metadata_standard=metadata_standard,
    metadata_standard_name="spatial_ecological"
)

2026-05-28 14:32:53,474 - INFO - root - PlanExecutor initialized with topology: single
2026-05-28 14:32:53,475 - INFO - root -   Players per step: 1
2026-05-28 14:32:53,475 - INFO - root -   Debate rounds: 0
2026-05-28 14:32:53,475 - INFO - root -   Player pool: ['data_analyst']
2026-05-28 14:32:53,475 - INFO - root - Using structured output schema: SpatialEcologicalMetadata
2026-05-28 14:32:53,475 - INFO - root - ============================================================
2026-05-28 14:32:53,476 - INFO - root - STARTING PLAN EXECUTION
2026-05-28 14:32:53,476 - INFO - root - Context: biota_multi
2026-05-28 14:32:53,476 - INFO - root - Type: multi_csv
2026-05-28 14:32:53,476 - INFO - root - Resources: ['biota', 'samples', 'species']
2026-05-28 14:32:53,476 - INFO - root - Steps: 4
2026-05-28 14:32:53,476 - INFO - root - ============================================================
2026-05-28 14:32:53,476 - INFO - root - 
2026-05-28 14:32:53,477 - INFO - root - ==================== STEP 

/home/com3dian/Github/metadata_agent/notebooks/../src/players/player.py:1150: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return Player(


2026-05-28 14:34:27,416 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-05-28 14:34:27,417 - INFO - root -   Player 'data_analyst_1' completed execution
2026-05-28 14:34:27,418 - INFO - root -     Output: ### 1. Approach to the Task  To perform a comprehensive field statistics and missing value analysis for the `biota_multi` context, I executed the following steps:  1.  **Context Ingestion**: Retrieved...
2026-05-28 14:34:27,418 - INFO - root - Single player, skipping debate
2026-05-28 14:34:27,419 - INFO - root - --- STEP 0: SYNTHESIS ---
2026-05-28 14:34:46,670 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-05-28 14:34:46,672 - INFO - root -   Synthesis complete. Produced artifacts: ['biota:field_stats', 'biota:missing_values', 'samples:field_stats', 'samples:missing_values', 'species:field_stats', 'species:missing_values']
2026-05-28 14:34:46,672 - INFO - root -     S

/home/com3dian/Github/metadata_agent/notebooks/../src/players/player.py:1150: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return Player(


2026-05-28 14:35:39,809 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-05-28 14:35:39,811 - INFO - root -   Player 'relationship_analyst_1' completed execution
2026-05-28 14:35:39,812 - INFO - root -     Output: ### 1. Approach to the Task  To discover and validate relationships between the resources `biota`, `samples`, and `species` in the `biota_multi` context, I employed the following analytical strategy: ...
2026-05-28 14:35:39,812 - INFO - root - Single player, skipping debate
2026-05-28 14:35:39,813 - INFO - root - --- STEP 1: SYNTHESIS ---
2026-05-28 14:35:49,743 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-05-28 14:35:49,745 - INFO - root -   Synthesis complete. Produced artifacts: ['relationships']
2026-05-28 14:35:49,746 - INFO - root -     Synthesized output: ### Validated Relationships for `biota_multi` Context  **1. Relationship: Biota to Samples** *   **

In [8]:
from pprint import pprint
pprint(result.final_workspace['metadata_output'])

{'description': 'Comprehensive dataset containing biological observations of '
                'benthic macrofauna in the Wadden Sea, including abundance and '
                'biomass metrics (AFDM), linked to sampling metadata and '
                'taxonomic species information. The dataset covers 391,018 '
                'observations across 51,851 samples and 177 species, primarily '
                'collected via boat using grid and random sampling strategies.',
 'format': 'CSV',
 'methods': 'Biological samples collected using grid (85%) and random (15%) '
            'sampling strategies, primarily from boats (92%). Samples '
            'processed to determine species abundance per square meter '
            '(abundance_m2) and ash-free dry mass per square meter (afdm_m2). '
            'Taxonomic identification linked via Sibes ID. Environmental '
            'variables (median grain size, percentage mud) recorded for most '
            'samples, though 7.1% of samples lack t

In [8]:
result.final_workspace

{'metadata_standard': '{\n    "title": "...",\n    "description": "...",\n    "subject": "...",\n    "spatial_coverage": "Geographic bounding box in WGS84 with numeric fields: min_lat, min_lon, max_lat, max_lon",\n    "spatial_resolution": "...",\n    "temporal_coverage": "Time period covered, from and to date",\n    "temporal_resolution": "Temporal resolution of the data, e.g. daily, monthly, yearly",\n    "methods": "Methods used for data collection",\n    "format": "..."\n}',
 'biota:stats': "### Data Resource Statistics and Quality Summary\n\n| Resource | Record Count | Key Characteristics | Data Quality/Completeness |\n| :--- | :--- | :--- | :--- |\n| **Biota** | 391,018 | Core transactional data; high variance in `abundance_m2` and `afdm_m2`. | 100% complete; no missing values. |\n| **Samples** | 3,665 | 505 unique dates; 12 tidal basins; primary sampling via 'grid' and 'boat'. | ~7% missingness in `median_grain_size` and `percentage_mud`. |\n| **Species** | 177 | Taxonomic diver